In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 54 — Nuanced Translation Tuner (SFT)

## Goal
Fine-tune for **domain-specific translation style** — e.g., medical English→Spanish
where standard translation misses terminology or tone.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Domain-specific SFT** | Teach a model a specialized translation register |
| **Style consistency** | Enforce formal/informal tone via training data |
| **Terminology adherence** | Fine-tune on domain glossary |

## Stack
- `transformers` + `peft` + `trl`
- Model: **Qwen2.5-0.5B-Instruct**

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Domain-Specific Translation Dataset

Medical patient instructions: English → Spanish.
Standard translators often use informal register or miss medical terms.
We want **formal usted-form Spanish** with correct terminology.

In [ ]:
translation_data = [
    {"en": "Take two tablets by mouth every 8 hours with food.",
     "es": "Tome dos comprimidos por vía oral cada 8 horas con alimentos."},
    {"en": "Do not stop taking this medication without consulting your doctor.",
     "es": "No deje de tomar este medicamento sin consultar a su médico."},
    {"en": "You may experience mild drowsiness. Avoid driving or operating heavy machinery.",
     "es": "Puede experimentar somnolencia leve. Evite conducir o manejar maquinaria pesada."},
    {"en": "Apply a thin layer of ointment to the affected area twice daily.",
     "es": "Aplique una capa fina de ungüento en el área afectada dos veces al día."},
    {"en": "If you experience chest pain, shortness of breath, or severe dizziness, seek emergency care immediately.",
     "es": "Si experimenta dolor en el pecho, dificultad para respirar o mareos intensos, busque atención de emergencia de inmediato."},
    {"en": "Your blood pressure reading today was 140/90 mmHg, which is considered stage 1 hypertension.",
     "es": "Su lectura de presión arterial hoy fue de 140/90 mmHg, lo cual se considera hipertensión en etapa 1."},
    {"en": "Please fast for 12 hours before your blood test tomorrow morning.",
     "es": "Por favor, ayune durante 12 horas antes de su análisis de sangre mañana por la mañana."},
    {"en": "The MRI results show no signs of a herniated disc. Your back pain is likely muscular.",
     "es": "Los resultados de la resonancia magnética no muestran signos de hernia discal. Su dolor de espalda probablemente sea muscular."},
    {"en": "We recommend scheduling a follow-up appointment in 4 weeks to monitor your cholesterol levels.",
     "es": "Le recomendamos programar una cita de seguimiento en 4 semanas para monitorear sus niveles de colesterol."},
    {"en": "Your child has been diagnosed with acute otitis media. We are prescribing amoxicillin for 10 days.",
     "es": "A su hijo/a se le ha diagnosticado otitis media aguda. Le estamos recetando amoxicilina durante 10 días."},
    {"en": "Avoid consuming grapefruit or grapefruit juice while taking this statin medication.",
     "es": "Evite consumir toronja o jugo de toronja mientras toma este medicamento con estatinas."},
    {"en": "The surgeon will make a small incision below your navel for the laparoscopic procedure.",
     "es": "El cirujano realizará una pequeña incisión debajo de su ombligo para el procedimiento laparoscópico."},
    {"en": "You need to take this antibiotic with a full glass of water to prevent esophageal irritation.",
     "es": "Debe tomar este antibiótico con un vaso lleno de agua para prevenir la irritación esofágica."},
    {"en": "Your hemoglobin A1C level is 7.2%, which indicates that your diabetes is not well controlled.",
     "es": "Su nivel de hemoglobina A1C es del 7,2%, lo cual indica que su diabetes no está bien controlada."},
    {"en": "Please remove all jewelry and metal objects before entering the MRI suite.",
     "es": "Por favor, retírese todas las joyas y objetos metálicos antes de entrar a la sala de resonancia magnética."},
    {"en": "After the procedure, you may experience mild cramping. This is normal and should subside within 24 hours.",
     "es": "Después del procedimiento, puede experimentar cólicos leves. Esto es normal y debería desaparecer dentro de las 24 horas."},
    {"en": "We are referring you to a cardiologist for further evaluation of your heart murmur.",
     "es": "Lo estamos refiriendo a un cardiólogo para una evaluación más detallada de su soplo cardíaco."},
    {"en": "Elevate the injured limb above heart level and apply ice for 20 minutes every 2 hours.",
     "es": "Eleve la extremidad lesionada por encima del nivel del corazón y aplique hielo durante 20 minutos cada 2 horas."},
]

train_data = translation_data[:15]
eval_data = translation_data[15:]
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

In [ ]:
from datasets import Dataset

SYS_MSG = (
    "You are a medical translator specializing in English to Spanish translation. "
    "Use formal register (usted form). Use correct medical terminology in Spanish. "
    "Translate accurately — do not add or omit information. Output only the translation."
)

formatted = []
for ex in train_data:
    formatted.append({"messages": [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Translate to Spanish:\n{ex['en']}"},
        {"role": "assistant", "content": ex["es"]},
    ]})

train_ds = Dataset.from_list(formatted)
print(f"Dataset: {train_ds}")

## Step 2 — Fine-Tune Translation Model

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./medical_translator_model"

trainer = SFTTrainer(
    model=MODEL_ID,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_steps=80,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=8,
        max_length=384,
        bf16=True,
        logging_steps=10,
        gradient_checkpointing=True,
        report_to="none",
    ),
    train_dataset=train_ds,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

print("Fine-tuning medical translator...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("✅ Done!")

## Step 3 — Test Translations

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def translate_medical(english_text):
    messages = [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Translate to Spanish:\n{english_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

# Test on held-out examples
for ex in eval_data:
    pred = translate_medical(ex["en"])
    print(f"EN: {ex['en']}")
    print(f"Expected: {ex['es']}")
    print(f"Got:      {pred}")
    print()

# Test on unseen medical text
new_texts = [
    "Take one capsule at bedtime. Do not crush or chew the extended-release tablet.",
    "Your white blood cell count is elevated, which may indicate an infection.",
]

print("=" * 60)
print("UNSEEN EXAMPLES:")
print("=" * 60)
for text in new_texts:
    print(f"\nEN: {text}")
    print(f"ES: {translate_medical(text)}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Domain glossary** | Medical terms mapped to correct Spanish equivalents |
| **Register enforcement** | Formal "usted" form via system prompt + training data |
| **Terminology accuracy** | "blood pressure" → "presión arterial", not casual alternatives |
| **Style consistency** | All translations follow same formal, clinical tone |

## 🔧 Production Extensions
- Add a bilingual medical glossary as context
- Use BLEU/METEOR metrics for quantitative evaluation
- Support multiple language pairs (EN→FR, EN→DE)
- Add back-translation validation (ES→EN→compare)